# **DASK Application**

**Version:** 1.0 | **Last updated:** 2026-08-11

**Author:** Asif Ashraf | **Author institution:** EarthScope Consortium

**Maintainer:** EarthScope OnRamp Team | **Maintainer's contact:** help@earthscope.org

**License:** CC-BY-4.0

---

### **Introduction**

This notebook demonstrates how `dask` can accelerate a real geophysics workflow. We will use `obspy` to retrieve seismic waveform data through EarthScope web services, remove the instrument response, and calculate simple waveform measurements for multiple seismic stations.

To understand where Dask fits into a scientific workflow, we will implement the same analysis in two different ways:

- **First**, process each station sequentially using a standard Python loop;
- **Second**, wrap the station-processing function with `dask.delayed` to build a parallel workflow;
- **Finally**, compare the serial and parallel execution times to see the performance benefits of parallel computing.

Although this example focuses on seismic waveform processing, the same pattern applies to most geophysical workflows. When the same sequence of operations is repeated, for example retrieving data, preprocessing it, performing an analysis, and saving the results, workflow can often be parallelized with Dask to reduce overall computation time.

### **Learning objectives**

By the end of this notebook, you will be able to:

- apply `dask` to a real seismic-data workflow;
- separate shared setup operations from station-level processing;
- create one `dask.delayed` task for each waveform request; and
- explain when the delayed workflow begins executing.

### **Table of Contents**

1. [Seismic workflow](#1-seismic-workflow)
2. [Waveform request](#2-waveform-request)
3. [Define the unit of work](#3-define-the-unit-of-work)
4. [Serial processing](#4-serial-processing)
5. [Parallel processing](#5-parallel-processing)
6. [Summary](#6-summary)

---

### **1. Seismic workflow**

Scientific workflows usually begin by defining the scope of the analysis. Here, we specify the earthquake origin time and location, the seismic network and channel of interest, and the waveform time window that will be retrieved.

For this exercise, we’ll focus on a 2019 earthquake in Seattle, Washington ([see earthquake details](https://earthquake.usgs.gov/earthquakes/eventpage/uw61535372/executive)), a magnitude 4.6 earthquake around 2 km south of Roosevelt, Washington. It's a great example because it has well-recorded data and is well covered by `UW`-network ([see network details](https://www.fdsn.org/networks/detail/UW/)) stations whose data are available through EarthScope web services.

The workflow will examine vertical-component broadband waveforms from stations in the `UW` network located within 100 km of the earthquake. For each available station, we will retrieve a window beginning 10 seconds before the earthquake origin time and ending 120 seconds after it. The event time, waveform window, network, and channel will be shared by every station-processing task.

The parameters used in this notebook tutorial are defined below.

In [8]:
from obspy import UTCDateTime

event_time = UTCDateTime("2019-07-12T09:51:38")

event_lat = 47.873
event_lon = -122.016

network = "UW"
channel = "HHZ"

pre_time = 10
post_time = 120
search_radius_km = 100

The cell above defines every shared parameter of the analysis in one place: the earthquake origin time and epicenter, the network (`UW`) and channel (`HHZ`, vertical-component broadband) to analyze, the waveform window (10 s before to 120 s after origin time), and the 100-km station search radius. Collecting these values here means every later step &mdash; the metadata query, the serial loop, and the Dask tasks &mdash; draws on the same definitions, so changing the experiment only requires editing this one cell.

### **2. Waveform request**

Before retrieving waveform data, we need to determine which seismic stations and channels were operating at the time of the earthquake. We use ObsPy's FDSN client to query the EarthScope station service and return an ObsPy `Inventory`.

The inventory request uses `level="response"` because the instrument-response metadata will later be needed to convert the recorded digital counts into physical ground-motion measurements.

This metadata query is a shared setup operation. It is performed once before the workflow separates into independent station-processing branches.

In [9]:
from obspy.clients.fdsn import Client as FDSNClient
metadata_client = FDSNClient("EARTHSCOPE")
inventory = metadata_client.get_stations(network=network,station="*", location="*",
                        channel=channel, latitude=event_lat, longitude=event_lon,
                        maxradius=search_radius_km / 111.2, 
                        starttime=event_time, endtime=event_time + 1,level="response")
inventory

Inventory created at 2026-07-31T20:52:32.256900Z
	Created by: EarthScope WEB SERVICE: fdsnws-station | version: 1.1.56
		    https://service.earthscope.org/fdsnws/station/1/query?starttime=201...
	Sending institution: EarthScope (EarthScope)
	Contains:
		Networks (1):
			UW
		Stations (13):
			UW.BERY (Pilchuck Tree Farm, Arlington, WA, USA)
			UW.BST16 (BEST site 16, Port Orchard, WA, USA)
			UW.BST19 (BEST site 19, Seabeck, WA, USA)
			UW.BST20 (BEST site 20, Silverdale, WA, USA)
			UW.BST21 (BEST site 21, Tahuya, WA, USA)
			UW.BST22 (BEST site 22, Belfair, WA, USA)
			UW.BST23 (BEST site 23, Lilliwaup, WA, USA)
			UW.DOSE (Dosewallips, Brinnon, WA, USA)
			UW.GNW (Green Mountain, WA, USA)
			UW.RATT (Rattlesnake Lake, King County, WA)
			UW.SP2 (Seward Park, Seattle, WA, USA)
			UW.STOR (Enumclaw, WA, USA)
			UW.TKEY (Lakebay, WA, USA)
		Channels (13):
			UW.BERY..HHZ, UW.BST16..HHZ, UW.BST19..HHZ, UW.BST20..HHZ, 
			UW.BST21..HHZ, UW.BST22..HHZ, UW.BST23..HHZ, UW.DOSE..HHZ, 
			UW

This cell creates an FDSN web-service client pointed at the **EarthScope** data center and asks its station service for all `NC`-network stations with the requested channel that were operating at the event time within the search radius (`maxradius` is in degrees, so we divide kilometers by ~111.2 km/degree). Because we pass `level="response"`, the returned `Inventory` includes full instrument-response metadata, which `process_station()` will later need to convert raw counts into physical ground motion. This single query is the shared setup step that every station branch depends on.

An ObsPy `Inventory` is hierarchical: it contains networks, stations, and the individual channels available at each station. To distribute the work with Dask, we will convert this nested structure into a simpler list of waveform requests.

Each request contains four identifiers:

- **network code**;
- **station code**;
- **location code**; and
- **channel code**.

Together, these values form the network–station–location–channel identifier.

In [10]:
requests = set()

for net in inventory:
    for sta in net:
        for cha in sta:
            requests.add(
                (
                    net.code,
                    sta.code,
                    cha.location_code or "",
                    cha.code,
                )
            )
# Convert the unique tuples into dictionaries.
requests = [
    {
        "network": net,
        "station": sta,
        "location": loc,
        "channel": cha,
    }
    for net, sta, loc, cha in sorted(requests)
]
requests

[{'network': 'UW', 'station': 'BERY', 'location': '', 'channel': 'HHZ'},
 {'network': 'UW', 'station': 'BST16', 'location': '', 'channel': 'HHZ'},
 {'network': 'UW', 'station': 'BST19', 'location': '', 'channel': 'HHZ'},
 {'network': 'UW', 'station': 'BST20', 'location': '', 'channel': 'HHZ'},
 {'network': 'UW', 'station': 'BST21', 'location': '', 'channel': 'HHZ'},
 {'network': 'UW', 'station': 'BST22', 'location': '', 'channel': 'HHZ'},
 {'network': 'UW', 'station': 'BST23', 'location': '', 'channel': 'HHZ'},
 {'network': 'UW', 'station': 'DOSE', 'location': '', 'channel': 'HHZ'},
 {'network': 'UW', 'station': 'GNW', 'location': '', 'channel': 'HHZ'},
 {'network': 'UW', 'station': 'RATT', 'location': '', 'channel': 'HHZ'},
 {'network': 'UW', 'station': 'SP2', 'location': '', 'channel': 'HHZ'},
 {'network': 'UW', 'station': 'STOR', 'location': '', 'channel': 'HHZ'},
 {'network': 'UW', 'station': 'TKEY', 'location': '', 'channel': 'HHZ'}]

The resulting `requests` list acts as the work queue for the remainder of the notebook. Each dictionary describes one waveform that can be retrieved and processed. Using a set first removes duplicate channel combinations, while sorting the values produces a consistent request order.

> **Key idea:** After the shared inventory has been retrieved, one waveform request does not depend on the result from another waveform request. The requests can therefore be processed independently.

### **3. Define the unit of work**

Before introducing Dask, we must decide what tasks should be accomplished. In this workflow, the unit of work is the complete processing sequence for one waveform request.

The `process_station()` function performs the following operations:

1. Create an ObsPy FDSN client.
2. Retrieve one waveform from EarthScope web services.
3. Merge adjacent waveform segments when necessary.
4. Detrend and taper the waveform.
5. Remove the instrument response.
6. Calculate the peak and root-mean-square amplitudes.
7. Return a small dictionary containing the results.

The operations within one station branch must remain sequential. For example, the instrument response cannot be removed before the waveform is retrieved. However, the complete branches for separate stations do not depend on one another and may therefore be executed in parallel by the Dask scheduler.

In [11]:
import numpy as np

def process_station(request, event_time, pre_time, post_time, inventory):
    """ Retrieve and process one seismic waveform using ObsPy """
    
    client = FDSNClient("EARTHSCOPE")
    
    try:
        stream = client.get_waveforms(
            network=request["network"],
            station=request["station"],
            location=request["location"],
            channel=request["channel"],
            starttime=event_time - pre_time,
            endtime=event_time + post_time,
        )
        
        print(f"Waveform found for station: {request['station']}")
            
        stream.merge(method=1, fill_value="interpolate")
    
        trace = stream[0].copy()
    
        trace.detrend("linear")
        trace.detrend("demean")
        trace.taper(max_percentage=0.05)
        
        trace.remove_response(inventory = inventory)
    
        # Calculate two simple waveform measurements.
        peak_amplitude = float(np.max(np.abs(trace.data)))
        rms_amplitude = float(np.sqrt(np.mean(np.square(trace.data))))
    
        waveform_id = ".".join(
            [
                request["network"],
                request["station"],
                request["location"],
                request["channel"],
            ]
        )
        return {
        "waveform_id":waveform_id,
        "peak_amplitude":peak_amplitude,
        "rms_amplitude":rms_amplitude
        }
    except Exception as e:
        # Surface the reason (no data, network error, ...) instead of failing silently.
        print(f"No waveform returned for station: {request['station']} ({type(e).__name__}: {e})")
        return None

This function can now be called in two different ways:

- directly from a normal Python loop for serial execution; or
- through `dask.delayed` for parallel execution.

The scientific processing steps do not need to be rewritten. Dask changes the execution strategy around the function rather than changing the underlying analysis.

### **4. Serial Processing**

In the following loop, Python processes the waveform requests one at a time:

1. retrieve and process the first waveform;
2. wait until that request finishes;
3. move to the next waveform; and
4. continue until every request has been attempted.

The `perf_counter()` function records the total elapsed time for the complete loop.

In [12]:
from time import perf_counter

serial_start = perf_counter()

serial_results = []
for request in requests:
    result = process_station(request=request, event_time=event_time,
                            pre_time=pre_time, post_time=post_time, 
                            inventory=inventory)
    # process_station() returns None when no waveform is available, so keep
    # only the successful results.
    if result is not None:
        serial_results.append(result)

serial_runtime = perf_counter() - serial_start
print(f"** Runtime for serial processing: {serial_runtime:.2f} s ({len(serial_results)} of {len(requests)} stations returned data) **")

Waveform found for station: BERY
Waveform found for station: BST16
Waveform found for station: BST19
Waveform found for station: BST20
Waveform found for station: BST21
Waveform found for station: BST22
Waveform found for station: BST23
Waveform found for station: DOSE
Waveform found for station: GNW
Waveform found for station: RATT
Waveform found for station: SP2
Waveform found for station: STOR
Waveform found for station: TKEY
** Runtime for serial processing: 5.71 s (13 of 13 stations returned data) **


This loop is the baseline: it calls `process_station()` directly, one request at a time, so each download must finish completely before the next begins. The elapsed time therefore approximates the *sum* of all the individual retrieval and processing times. Failed requests return `None` and are skipped, so `serial_results` contains one dictionary of measurements per successfully retrieved waveform. Note the runtime printed here &mdash; we will compare it against the parallel version next.

### **5. Parallel Processing**

We will now apply Dask Delayed to the same `process_station()` function. The processing logic remains unchanged; only the way the function calls are represented and executed will change.

For each waveform request, `delayed(process_station)(...)` creates a lazy Dask task. At this stage, Dask records:

- the function that should be called;
- the input arguments for that request; and
- the dependencies associated with the task.

In [ ]:
from dask.distributed import Client

client = Client(
    n_workers=2,
    threads_per_worker=2
)

client

**Where does this computation actually run? Local scheduler vs. distributed cluster**

Note that this notebook never starts a Dask cluster or creates a `dask.distributed.Client`. When `compute()` is called without a client, Dask falls back to its default **local threaded scheduler**: the tasks run in a thread pool inside this notebook's own Python process, on this one machine &mdash; not on distributed workers.

That is a deliberate and reasonable choice here. The dominant cost of each task is network I/O (waiting for EarthScope web services to return waveforms), and ObsPy releases Python's Global Interpreter Lock (GIL) while waiting on the network, so multiple threads genuinely overlap their downloads. Threads also share memory, so the `inventory` object is passed to every task without any serialization cost.

As a rule of thumb:

- **Local threaded scheduler** (what we use here) &mdash; best when tasks are I/O-bound or call GIL-releasing libraries (NumPy, ObsPy), the data fits on one machine, and simplicity matters. Zero setup, minimal overhead.
- **Local `LocalCluster` + `Client`** (as in the companion *Dask Operation* notebook) &mdash; still one machine, but with worker *processes*, the diagnostic dashboard, and Futures support. Useful when tasks are CPU-bound pure-Python code that would otherwise serialize on the GIL, or when you want to watch execution live.
- **Distributed cluster** (e.g., a Dask Gateway cluster on GeoLab) &mdash; needed when the computation or the data exceeds one machine: thousands of waveforms, terabyte-scale arrays, or many-node scaling. The workflow code below would not change; only the client connection would.

Because the code is identical in all three cases, a good practice &mdash; and the one this tutorial follows &mdash; is to develop and debug locally, then point the same workflow at a larger cluster only when the problem size demands it.

In [13]:
from dask import delayed 

station_tasks = []

for request in requests:
    task = delayed(process_station)(request=request, event_time=event_time,
                                    pre_time=pre_time, post_time=post_time,
                                    inventory=inventory)
    station_tasks.append(task)

workflow = delayed(list)(station_tasks)

parallel_start = perf_counter()

parallel_results = workflow.compute()

# Filter out failed requests, exactly as in the serial loop.
parallel_results = [r for r in parallel_results if r is not None]

parallel_runtime = perf_counter() - parallel_start

print(f"** Runtime for parallel processing: {parallel_runtime:.2f} s ({len(parallel_results)} of {len(requests)} stations returned data) **")

Waveform found for station: BST23
Waveform found for station: BST19
Waveform found for station: TKEY
Waveform found for station: RATT
Waveform found for station: STOR
Waveform found for station: BERY
Waveform found for station: DOSE
Waveform found for station: BST20
Waveform found for station: GNW
Waveform found for station: BST16
Waveform found for station: SP2
Waveform found for station: BST21
Waveform found for station: BST22
** Runtime for parallel processing: 1.27 s (13 of 13 stations returned data) **


The loop body looks almost identical to the serial version &mdash; the only change is that each call is wrapped in `delayed()`, so instead of executing, it records a task. `delayed(list)(station_tasks)` adds one final task that gathers every station result into a single list, and nothing runs until `workflow.compute()` is called. At that point the scheduler executes the independent station tasks concurrently in a thread pool, so the elapsed time is set by the *slowest* downloads rather than the sum of all of them &mdash; which is why the printed runtime should be substantially shorter than the serial one.

| Serial workflow | Dask Delayed workflow |
|---|---|
| Calls the function immediately | Creates lazy task objects |
| Processes requests one at a time | May process independent requests concurrently |
| Uses an ordinary Python loop | Uses a Dask task graph |
| Produces each result during the loop | Produces the final list when `compute()` finishes |
| Has little scheduling overhead | Requires task-scheduling overhead |

In [14]:
# -- Compare the two runtimes side by side --

speedup = serial_runtime / parallel_runtime

print(f"Serial runtime:    {serial_runtime:6.2f} s")
print(f"Parallel runtime:  {parallel_runtime:6.2f} s")
print(f"Speedup:           {speedup:6.2f}x  using the local threaded scheduler")

Serial runtime:      5.71 s
Parallel runtime:    1.27 s
Speedup:             4.49x  using the local threaded scheduler


### **6. Summary**

In this notebook we accelerated a real seismic workflow with Dask while leaving the science code untouched. Looking back at the learning objectives:

- **Applying Dask to a real workflow** &mdash; the same `process_station()` function ran serially and in parallel; only the execution style changed.
- **Separating shared setup from station-level work** &mdash; the single `level="response"` inventory query ran once, up front, and was shared by every task.
- **One `delayed` task per request** &mdash; each waveform request became an independent, lazy task in the graph.
- **When execution begins** &mdash; nothing ran until `workflow.compute()`, at which point the local threaded scheduler executed the independent tasks concurrently.

**When does this pattern pay off?** It shines when a workflow consists of *many independent, I/O-bound tasks* &mdash; here, dozens of waveform downloads that spend most of their time waiting on the network. The benefit shrinks when there are only a few tasks, when tasks depend heavily on one another (limiting concurrency), or when per-task work is so small that scheduling overhead dominates.

**Next steps.** The same workflow, unchanged, can run on a distributed Dask Gateway cluster on GeoLab: connect a `Client` to the Gateway cluster and call `workflow.compute()` as before. The companion *Dask Operation* notebook shows the `Client`-based setup, the diagnostic dashboard, and the Futures interface for dynamic workflows. (If you do create a `Client`, remember to call `client.close()` when finished &mdash; this releases shared GeoLab resources for other users.)